# 🔍 Hybrid Retrieval: FAISS + BM25 + RRF
**Mata Kuliah:** Natural Language Processing (UAS)

Notebook ini membangun sistem retrieval hybrid untuk dataset hukum Klinik Hukumonline.
* **Dense Retrieval:** FAISS `IndexFlatIP` dengan vektor `multilingual-e5-large` (1024-dim, L2-normalized)
* **Sparse Retrieval:** BM25 berbasis teks `title + question + summarize`
* **Fusion:** Reciprocal Rank Fusion (RRF, k=60)
* **Input:** `vectors.npy` dan `documents.json` dari `embedding.ipynb`

## 1. Install Dependensi

In [14]:
!pip install -q faiss-cpu rank-bm25 sentence-transformers torch numpy

## 2. Import & Muat Data

In [15]:
import os
import json
import numpy as np
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

# Auto-detect lokasi file (lokal atau Colab)
VECTORS_PATH = "vectors.npy"
DOCS_PATH    = "documents.json"

assert os.path.exists(VECTORS_PATH), f"File tidak ditemukan: {VECTORS_PATH}"
assert os.path.exists(DOCS_PATH),    f"File tidak ditemukan: {DOCS_PATH}"

vectors = np.load(VECTORS_PATH).astype(np.float32)
with open(DOCS_PATH, "r", encoding="utf-8") as f:
    docs = json.load(f)

print(f"Vectors shape : {vectors.shape}")   # (7172, 1024)
print(f"Docs count    : {len(docs)}")        # 7172
print(f"Norm sample   : {np.linalg.norm(vectors[0]):.4f}")  # ~1.0

Vectors shape : (7172, 1024)
Docs count    : 7172
Norm sample   : 1.0000


## 3. Muat Model E5 (untuk Encode Query)

In [16]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device.upper()}")

model = SentenceTransformer("intfloat/multilingual-e5-large", device=device)
print("Model berhasil dimuat!")

Device: CUDA


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model berhasil dimuat!


## 4. Build FAISS Index (Dense Retrieval)
`IndexFlatIP` = Inner Product (sama dengan cosine similarity karena vektor sudah L2-normalized)

In [17]:
dim = vectors.shape[1]  # 1024
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(vectors)

print(f"FAISS index built: {faiss_index.ntotal} vectors, dim={dim}")

FAISS index built: 7172 vectors, dim=1024


## 5. Build BM25 Index (Sparse Retrieval)
Teks yang diindeks: `title + question + summarize` (konsisten dengan field yang di-embed)

In [18]:
# Tokenisasi sederhana: lowercase + split whitespace
bm25_corpus = [
    (doc["title"] + " " + doc["question"] + " " + doc["summarize"]).lower().split()
    for doc in docs
]

bm25_index = BM25Okapi(bm25_corpus)
print(f"BM25 index built: {len(bm25_corpus)} dokumen")

BM25 index built: 7172 dokumen


## 6. Fungsi Hybrid Search (FAISS + BM25 + RRF)
**Reciprocal Rank Fusion (RRF):** `score = 1/(k + rank_dense) + 1/(k + rank_sparse)` dengan `k=60`

In [19]:
def hybrid_search(query: str, top_k: int = 5, rrf_k: int = 60) -> list[dict]:
    """
    Hybrid retrieval: FAISS (dense) + BM25 (sparse) fused dengan RRF.
    
    Args:
        query   : Teks pertanyaan dari user
        top_k   : Jumlah dokumen yang dikembalikan
        rrf_k   : Konstanta RRF (default=60, standar dari paper Cormack et al.)
    
    Returns:
        List of dict: dokumen terurut berdasarkan skor RRF (descending)
    """
    # --- Dense retrieval (FAISS) ---
    q_vec = model.encode(
        f"query: {query.strip()}",
        normalize_embeddings=True,
        convert_to_numpy=True
    ).reshape(1, -1).astype(np.float32)

    candidate_pool = min(top_k * 10, len(docs))  # ambil lebih banyak kandidat untuk di-fuse
    dense_scores, dense_ids = faiss_index.search(q_vec, candidate_pool)
    dense_ids = dense_ids[0].tolist()           # list of doc indices, ranked by score

    # --- Sparse retrieval (BM25) ---
    bm25_scores = bm25_index.get_scores(query.lower().split())
    sparse_ids  = np.argsort(bm25_scores)[::-1][:candidate_pool].tolist()

    # --- RRF Fusion ---
    rrf_scores: dict[int, float] = {}

    for rank, doc_id in enumerate(dense_ids):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (rrf_k + rank + 1)

    for rank, doc_id in enumerate(sparse_ids):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (rrf_k + rank + 1)

    # Urutkan berdasarkan skor RRF (descending), ambil top_k
    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    results = []
    for doc_id, rrf_score in ranked:
        doc = docs[doc_id].copy()
        doc["rrf_score"] = round(rrf_score, 6)
        doc["dense_score"] = round(float(np.dot(vectors[doc_id], q_vec[0])), 4)
        results.append(doc)

    return results


print("Fungsi hybrid_search() siap digunakan.")

Fungsi hybrid_search() siap digunakan.


## 7. Uji Coba Retrieval

In [20]:
def print_results(query: str, top_k: int = 3):
    results = hybrid_search(query, top_k=top_k)
    print(f"=== Query: '{query}' ===\n")
    for i, doc in enumerate(results, 1):
        print(f"[{i}] RRF={doc['rrf_score']:.5f}  Dense={doc['dense_score']:.4f}")
        print(f"     Judul: {doc['title']}")
        print(f"     Kasus: {doc['question'][:120]}...")
        print(f"     URL  : {doc['source']}\n")


# --- Test queries ---
print_results("bagaimana cara membeli tanah girik tanpa sertifikat yang aman?")

=== Query: 'bagaimana cara membeli tanah girik tanpa sertifikat yang aman?' ===

[1] RRF=0.03279  Dense=0.8989
     Judul: Kiat-kiat Membeli Tanah Girik yang Aman
     Kasus: Bagaimana cara pengurusan pembelian tanah girik (tanpa sertifikat hak milik) sesuai peraturan?...
     URL  : https://www.hukumonline.com/klinik/a/kiat-kiat-membeli-tanah-girik-yang-aman-cl1360/

[2] RRF=0.03077  Dense=0.8600
     Judul: Pengaturan Soal Tanah Bengkok
     Kasus: Kami mau tanya; 1. Siapakah yang membuat peraturan/UU tentang aset tanah desa ("bengkok")? 2. Apakah ada undang-undang/p...
     URL  : https://www.hukumonline.com/klinik/a/masalah-bengkok-desa-cl1420/

[3] RRF=0.03042  Dense=0.8689
     Judul: Tanah Warisan Belum Bersertifikat, Begini Cara Mengurusnya
     Kasus: Kami memiliki sebidang tanah yang berasal dari tanah warisan. Namun, tanah tersebut belum mempunyai surat-surat kepemili...
     URL  : https://www.hukumonline.com/klinik/a/tanah-warisan-belum-bersertifikat--begini-cara-mengurusn

In [21]:
print_results("bos gamau cairin uang lembur lapor kemana ya?")

=== Query: 'bos gamau cairin uang lembur lapor kemana ya?' ===

[1] RRF=0.01639  Dense=0.8067
     Judul: Hukumnya Meminta Uang Tebusan Binatang Piaraan Milik Orang Lain
     Kasus: Gan, ane mohon tanya. Sebelumnya ane minta maaf kalau sekiranya pertanyaan ane kurang berbobot atau tidak masuk kategori...
     URL  : https://www.hukumonline.com/klinik/a/hukumnya-meminta-uang-tebusan-binatang-piaraan-milik-orang-lain-lt4f76a86381246/

[2] RRF=0.01639  Dense=0.7844
     Judul: Dana BOS Disalahgunakan Sekolah, Ini Sanksinya
     Kasus: Seperti yang kita ketahui Bersama, Dana BOS yang ditujukan untuk setiap sekolah guna memperingan orang tua/wali murid se...
     URL  : https://www.hukumonline.com/klinik/a/dana-bos-disalahgunakan-sekolah--ini-sanksinya-lt54310ff037f2b/

[3] RRF=0.01613  Dense=0.8054
     Judul: Kapan Seharusnya Uang Pisah Cair?
     Kasus: Kapankah idealnya uang pisah/UPH/UPMK dibayarkan oleh perusahaan saat kita mengundurkan diri dari perusahaan tersebut? A...
     URL  : 

In [22]:
print_results("duit kantor kepake sama temen tapi udah dicicil balikin apa tetep dipenjara?")

=== Query: 'duit kantor kepake sama temen tapi udah dicicil balikin apa tetep dipenjara?' ===

[1] RRF=0.02751  Dense=0.8174
     Judul: Jika Dijebak oleh Polisi dan Disiksa dalam Tahanan
     Kasus: Teman saya kena masalah dan tambah banyak ketika dalam sel tahanan. Masalahnya begini, teman saya punya tetangga yang pe...
     URL  : https://www.hukumonline.com/klinik/a/jika-dijebak-oleh-polisi-dan-disiksa-dalam-tahanan-lt4cd22340054c7/

[2] RRF=0.02288  Dense=0.7885
     Judul: Dituduh Menggelapkan Booking Fee
     Kasus: Saya kerja di marketing properti. Kemarin saya jual rumah dengan harga yang disetujui kantor sebesar Rp360.000.000 dan k...
     URL  : https://www.hukumonline.com/klinik/a/dituduh-menggelapkan-booking-fee-lt4f165f3846c9b/

[3] RRF=0.01639  Dense=0.7712
     Judul: Jika Menjual Handphone Bekas Seolah-olah dalam Keadaan Baru
     Kasus: Adakah pidananya jika ada toko handphone second tapi jual handphone lama dengan keadaan kelihatan baru. Udah gitu dia me...
     URL 

In [23]:
print_results("how can a foreigner legally own property or land in bali?")

=== Query: 'how can a foreigner legally own property or land in bali?' ===

[1] RRF=0.03202  Dense=0.8549
     Judul: Foreign Worker Positions that Can Obtain KITAP
     Kasus: I am a foreigner residing in Bali. My question is, is there a regulation that explains what positions can be occupied by...
     URL  : https://www.hukumonline.com/klinik/a/foreign-worker-positions-that-can-obtain-kitap-lt65367961b598a/

[2] RRF=0.03154  Dense=0.8453
     Judul: The Right of Children of Mixed Marriages to Own a House
     Kasus: I am an Indonesian Citizen and my wife is a foreigner. When we agreed to buy a house through the mortgage loan program, ...
     URL  : https://www.hukumonline.com/klinik/a/the-right-of-children-of-mixed-marriages-to-own-a-house-lt64928f7a2dee1/

[3] RRF=0.03126  Dense=0.8531
     Judul: The Law if a Land with Right-to-Use Owned by Foreigners is Pledged as Collateral
     Kasus: What is hak pakai and right over land? What is the law on the right to use land owned by a fo